In [1]:
import sqlite3
import pandas as pd

In [ ]:
# Create a new SQLite database
DATABASE_NAME = "Outputs/employees.db"
def get_connection():
    return sqlite3.connect(DATABASE_NAME)

conn = get_connection()

# Create employees table if it does not exist
create_employees_table = """
CREATE TABLE IF NOT EXISTS employees (
    id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT,
    position TEXT,
    department TEXT,
    salary INTEGER,
    superior_id INTEGER,
    FOREIGN KEY (superior_id) REFERENCES employees(id)
);
"""

# Insert sample data into employees table
insert_employees_data = """
INSERT INTO employees (id, first_name, last_name, position, department, salary, superior_id) VALUES
(1, 'Alice', 'Smith', 'CEO', 'Management', 150000, NULL),
(2, 'Bob', 'Johnson', 'CTO', 'Technology', 120000, 1),
(3, 'Charlie', 'Brown', 'CFO', 'Finance', 120000, 1),
(4, 'David', 'Wilson', 'Engineering Manager', 'Technology', 90000, 2),
(5, 'Eve', 'Davis', 'Finance Manager', 'Finance', 90000, 3),
(6, 'Frank', 'Miller', 'Software Engineer', 'Technology', 80000, 4),
(7, 'Grace', 'Lee', 'Software Engineer', 'Technology', 80000, 4),
(8, 'Hannah', 'White', 'Accountant', 'Finance', 70000, 5),
(9, 'Ian', 'Clark', 'Data Scientist', 'Technology', 95000, 2),
(10, 'Jack', 'Adams', 'ML Engineer', 'Technology', 100000, 2);
"""

try:
    cursor = conn.cursor()
    cursor.execute(create_employees_table)
    conn.commit()
    cursor.execute("SELECT COUNT(*) FROM employees;")
    if cursor.fetchone()[0] == 0:
        cursor.execute(insert_employees_data)
        conn.commit()
except Exception as e:
    print("Error setting up employees table:", e)
finally:
    conn.close()

In [8]:
def run_query(query):
    conn = get_connection()
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df

query="SELECT * FROM employees LIMIT 5;"

run_query(query)

,id,first_name,last_name,position,department,salary,superior_id
0,1,Alice,Smith,CEO,Management,150000,NaN
1,2,Bob,Johnson,CTO,Technology,120000,1.0
2,3,Charlie,Brown,CFO,Finance,120000,1.0
3,4,David,Wilson,Engineering Manager,Technology,90000,2.0
4,5,Eve,Davis,Finance Manager,Finance,90000,3.0


In [6]:
# Example of a Non-Recursive CTE using employees
def execute_non_recursive_cte():
    conn = get_connection()
    query_non_recursive = """
    WITH employee_info AS (
      SELECT id, first_name || ' ' || last_name AS full_name, position, department, salary
      FROM employees
    )
    SELECT e.id, e.full_name, e.position, e.department, e.salary
    FROM employee_info e;
    """
    try:
        df_non_recursive = pd.read_sql_query(query_non_recursive, conn)
        print("Non-Recursive CTE Output:")
        print(df_non_recursive.head())
    except Exception as e:
        print("Error executing non-recursive CTE query:", e)
    finally:
        conn.close()

execute_non_recursive_cte()

Non-Recursive CTE Output:
   id      full_name             position  department  salary
0   1    Alice Smith                  CEO  Management  150000
1   2    Bob Johnson                  CTO  Technology  120000
2   3  Charlie Brown                  CFO     Finance  120000
3   4   David Wilson  Engineering Manager  Technology   90000
4   5      Eve Davis      Finance Manager     Finance   90000


In [13]:
# Example of a Recursive CTE to show hierarchy
def execute_recursive_cte():
    conn = get_connection()
    query_recursive = """
    WITH hierarchy AS (
      SELECT id, first_name, last_name, position, department, superior_id, salary, 1 AS level
      FROM employees
      WHERE superior_id IS NULL
      UNION ALL
      SELECT e.id, e.first_name, e.last_name, e.position, e.department, e.superior_id, e.salary, h.level + 1
      FROM employees e
      JOIN hierarchy h
      ON e.superior_id = h.id
    )
    SELECT last_name,  position, department, superior_id, salary,level FROM hierarchy ORDER BY level, superior_id;
    """
    try:
        df_recursive = pd.read_sql_query(query_recursive, conn)
        print("Recursive CTE Output:")
        print(df_recursive.head())
    except Exception as e:
        print("Error executing recursive CTE query:", e)
    finally:
        conn.close()

execute_recursive_cte()


Recursive CTE Output:
  last_name             position  department  superior_id  salary  level
0     Smith                  CEO  Management          NaN  150000      1
1   Johnson                  CTO  Technology          1.0  120000      2
2     Brown                  CFO     Finance          1.0  120000      2
3    Wilson  Engineering Manager  Technology          2.0   90000      3
4     Clark       Data Scientist  Technology          2.0   95000      3
